In [63]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [64]:
from broharness.llms.bedrock import bedrock, UserMessage, AIMessage, SystemMessage
from broharness.flows.skill_call import SkillCall
from broharness.flows.tool_call import ToolCall
from broharness.flows.tool_use import ToolUse
from broharness.flows.ask_user_question import AskUserQuestion
from broharness.flows.fail_recovery import FailRecovery
from broharness.flows.answer import Answer
from broflow import BaseTask, TaskRegistry, Flow
from pathlib import Path
import yaml
import sys
import subprocess
from broskill import SkillControl, ToolControl
from functools import partial
from broharness.toolblock import (
    tool_to_yaml, 
    load_skill_tool, 
    load_skill_extension_tool, 
    load_tool_tool, 
    ask_user_question_tool
)
from broharness.codeblock import parse_json_codeblock
from broharness.data_model import Process, State, LLMUse


ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

sc = SkillControl(SKILL_DIR)
sc.list_skills()
tc = ToolControl(sc)

In [65]:
from broskill.processing.tool import to_args

In [66]:
# scripts are now auto-registered when their skill loads -- manual load_tool
# is only needed as a fallback, e.g. for a script that failed to auto-register.
# tc.load_tool('file-ops', 'scripts/read_file.py')

In [67]:
TOOLS = dict(
    load_skill=sc.load_skill,
    load_skill_extension=sc.load_skill_extension,
    load_tool=tc.load_tool,
)

In [68]:
skill_call = SkillCall(name='skill-call', llm=bedrock, system_prompt='')
tool_call = ToolCall(name='tool-call', llm=bedrock, system_prompt='')
tool_use = ToolUse(name='tool-use', llm=bedrock, system_prompt='')
ask_user_question = AskUserQuestion(name='ask-user-question', llm=bedrock, system_prompt='')
fail_recovery = FailRecovery(name='fail-recovery', llm=bedrock, system_prompt='')
answer = Answer(name='answer', llm=bedrock, system_prompt='')

In [69]:
registry = TaskRegistry()
registry.register(Process.SKILL_CALL, skill_call)
registry.register(Process.TOOL_CALL, tool_call)
registry.register(Process.TOOL_USE, tool_use)
registry.register(Process.ASK_USER_QUESTION, ask_user_question)
registry.register(Process.FAIL_RECOVERY, fail_recovery)
registry.register(Process.ANSWER, answer)

flow = Flow(registry)

In [70]:
system_prompt = "You're Andy who is the best bro in the world. Always response in bro-tone with chill and mellow manner."

In [71]:
# this trigger load_skill with skill_name='tell-jokes'
# content = "tell me some jokes."
# this trigger ask_user_question
# content = "What's the capital of France?"
# this trigger nothing
# content = "1+1 is?"
# this trigger read_file
# content = "what is in skills/tell-joke/SKILL.md?"
# content = "What's in `one-liner.md`"
# this trigger list_directory, shallow (top-level folders under skills/)
# content = "what folders are directly under skills/?"
# this trigger list_directory, recursive (everything under skills/)
# content = "list everything under skills/, including subfolders."
# this trigger create_file (new file, parent dir exists)
content = "create a file called scratch/note.md with the content 'hello from the harness'"
# this trigger create_file -> already-exists error -> self-correct to update_file
# content = "create a file called scratch/note.md with the content 'second attempt'"
# this trigger update_file, but only after ask_user_question confirms the overwrite
# content = "update scratch/note.md to say 'updated content'"
# this trigger delete_file, but only after ask_user_question confirms the delete
# content = "delete scratch/note.md"

messages = [UserMessage(content)]
state = State(
    root=ROOT,
    skill_dir=SKILL_DIR,
    messages=messages,
    session_messages=messages.copy(),
    system_prompt=system_prompt or '',
    skill_control=sc,
    tool_control=tc,
    tools=TOOLS,
    session_tools=TOOLS.copy(),
    debug=messages.copy()
)

_ = flow.run(start=Process.SKILL_CALL, end=Process.END, state=state)

D:\study-on-agent\src\broharness\flows\skill_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill', 'input': {'skill_name': 'file-ops'}}]
load_skill passed
auto-registered create_file
auto-registered delete_file
auto-registered list_directory
auto-registered read_file
auto-registered update_file
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'create_file', 'input': {'path': 'scratch/note.md', 'content': 'hello from the harness'}}]
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\answer.py


In [72]:
state.messages

[{'role': 'user',
  'content': [{'text': "create a file called scratch/note.md with the content 'hello from the harness'"}]},
 {'role': 'assistant',
  'content': [{'text': 'Okay, I\'ve created a file named `scratch/note.md` and added the content "hello from the harness" to it.'}]}]

In [73]:
print(state.messages[-1]['content'][0]['text'])

Okay, I've created a file named `scratch/note.md` and added the content "hello from the harness" to it.


In [74]:
state.session_messages

[{'role': 'user',
  'content': [{'text': "create a file called scratch/note.md with the content 'hello from the harness'"}]},
 {'role': 'assistant',
  'content': [{'text': 'Okay, I\'ve created a file named `scratch/note.md` and added the content "hello from the harness" to it.'}]}]

In [75]:
state.registered_skills

{'file-ops': '# File Operations\n\n## Instructions\n\n### Reading and listing\n\n- If you already know the exact file the user means, use `scripts/read_file.py` with a\n  glob pattern narrow enough to match exactly that one file.\n- If the user names just a filename with no path (e.g. "what\'s in one-liner.md"), don\'t\n  guess its directory -- search for it anywhere in the project with `**/<filename>`\n  (e.g. `**/one-liner.md`). This isn\'t a guess, it\'s an exact-name search; if more than\n  one file shares that name, `read_file.py`\'s normal ambiguous-match handling applies.\n- If `scripts/read_file.py`\'s pattern matches more than one file, don\'t pick one\n  yourself even if it looks obvious -- call `ask_user_question` with the candidate\n  list from the error message and let the user choose.\n- If the user wants to see many files at once, or you aren\'t sure which single file they\n  mean, use `scripts/list_directory.py` first to see what matches a broader pattern, then\n  narro

In [76]:
print('\n\n'.join(state.extension_skills.values()))

In [77]:
state.session_tools

{'load_skill': <bound method SkillControl.load_skill of <broskill.processing.skill.SkillControl object at 0x000002688874DAC0>>,
 'load_skill_extension': <bound method SkillControl.load_skill_extension of <broskill.processing.skill.SkillControl object at 0x000002688874DAC0>>,
 'load_tool': <bound method ToolControl.load_tool of <broskill.processing.tool.ToolControl object at 0x0000026887B8B8C0>>}

In [78]:
state.error_message

''

In [79]:
state.debug

[{'role': 'user',
  'content': [{'text': "create a file called scratch/note.md with the content 'hello from the harness'"}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "load_skill",\n      "input": {\n        "skill_name": "file-ops"\n      }\n    }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 1079, 'outputTokens': 57, 'totalTokens': 1136}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "create_file",\n      "input": {\n        "path": "scratch/note.md",\n        "content": "hello from the harness"\n      }\n    }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 3880, 'outputTokens': 68, 'totalTokens': 3948}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "create_file", "input": { "path": "scratch/note.md", "content": "hello from the harness" } }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 3900, 'outputTokens': 56, 'totalTokens': 3956}},
 {'rol

In [80]:
state.tool_results

[{'create_file': 'created: scratch/note.md\n'}]

In [81]:
for ts in state.tool_results:
    for k, v in ts.items():
        print(f'{k}: {v}')

create_file: created: scratch/note.md



In [82]:
state.usage

{'SKILL_CALL': {'google.gemma-3-12b-it': {'input_tokens': 1079,
   'output_tokens': 57,
   'call_count': 1}},
 'TOOL_CALL': {'google.gemma-3-12b-it': {'input_tokens': 7780,
   'output_tokens': 124,
   'call_count': 2}},
 'ANSWER': {'google.gemma-3-12b-it': {'input_tokens': 2431,
   'output_tokens': 30,
   'call_count': 1}}}

In [83]:
def cal_usage(input, output, input_price, output_price, currency=1, session=1):
    mil = 1_000_000
    inputPrice = (input/mil)*input_price*currency*session
    outputPrice = (output/mil)*output_price*currency*session
    return inputPrice, outputPrice, inputPrice+outputPrice

In [84]:
in_p, out_p, currentcy, turn = 0.09, 0.29, 34, 1
for task, model_id in state.usage.items():
    print(task)
    for m_id, u in model_id.items():
        input_token = u['input_tokens']
        output_token = u['output_tokens']
        i, o, t = cal_usage(input_token, output_token, in_p, out_p, currentcy, turn)
        print(m_id)
        print(f"{i:.4f}, {o:.4f}, {t:.4f}")

SKILL_CALL
google.gemma-3-12b-it
0.0033, 0.0006, 0.0039
TOOL_CALL
google.gemma-3-12b-it
0.0238, 0.0012, 0.0250
ANSWER
google.gemma-3-12b-it
0.0074, 0.0003, 0.0077
